# Skin Disease Classification using Machine Learning
### Course-Based Project in Machine Learning & Neural Networks

**Objective:** Build an interpretable, highly accurate machine learning pipeline to classify six chronic dermatological diseases using clinical and histopathological observations from the UCI Dermatology Dataset.

**Target Diseases:**
1. Psoriasis
2. Seborrheic Dermatitis
3. Lichen Planus
4. Pityriasis Rosea
5. Chronic Dermatitis
6. Pityriasis Rubra Pilaris

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Data Loading & Preprocessing
- Load the UCI Dermatology Dataset (366 records, 34 attributes).
- Replace '?' missing values with NA and convert all features to numeric.
- Drop rows with missing values.

In [2]:
data_path = "../data/dermatology.csv" if os.path.exists("../data/dermatology.csv") else "data/dermatology.csv"
data = pd.read_csv(data_path)

# Handle missing values marked by '?'
data.replace('?', pd.NA, inplace=True)
data = data.apply(pd.to_numeric, errors='coerce')
data.dropna(inplace=True)

print(f"Dataset shape after cleaning: {data.shape}")
data.head()

## 2. Feature & Target Separation

In [3]:
X = data.iloc[:, :-1]
y = data.iloc[:, -1].astype(int)

disease_names = {
    1: "Psoriasis",
    2: "Seborrheic Dermatitis",
    3: "Lichen Planus",
    4: "Pityriasis Rosea",
    5: "Chronic Dermatitis",
    6: "Pityriasis Rubra Pilaris"
}

print(f"Feature matrix X: {X.shape}")
print(f"Target vector y:  {y.shape}")
print("Class distribution:\n", y.value_counts().sort_index())

## 3. Model Training & 5-Fold Cross Validation
Evaluate model stability using 5-Fold Stratified Cross Validation to avoid variance across train-test splits.

In [4]:
rf_cv = RandomForestClassifier(n_estimators=100, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_cv, X, y, cv=skf, scoring='accuracy')

print(f"5-Fold Accuracies: {[round(s, 4) for s in cv_scores]}")
print(f"Mean CV Accuracy:  {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 4. Train-Test Split & Evaluation

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=[disease_names[i] for i in sorted(y.unique())]))

## 5. Confusion Matrix & Diagnostic Feature Importance

In [6]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[disease_names[i] for i in sorted(y.unique())], yticklabels=[disease_names[i] for i in sorted(y.unique())], ax=ax)
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted Disease")
ax.set_ylabel("True Disease")
plt.xticks(rotation=30, ha="right")
plt.show()

In [7]:
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 5))
plt.title("Top 10 Diagnostic Features (Random Forest)")
plt.barh(range(10), importances[indices][::-1], align="center", color="#3b82f6")
plt.yticks(range(10), [X.columns[i] for i in indices][::-1])
plt.xlabel("Relative Importance")
plt.show()

## 6. Sample Prediction Testing

In [8]:
# Sample patient test
sample_input = np.array([
    2, 3, 2, 1, 1, 0, 0, 0, 2, 3, 1, 0, 1, 3, 0, 2,
    3, 3, 2, 1, 2, 1, 0, 0, 0, 2, 1, 2, 0, 0, 0, 2, 1, 35
]).reshape(1, -1)

pred = clf.predict(sample_input)[0]
print(f"Predicted Class: {pred} -> {disease_names[pred]}")

## 7. Model Serialization

In [9]:
model_export_path = "../model.pkl" if os.path.exists("../data") else "model.pkl"
joblib.dump(clf, model_export_path)
print(f"Model serialized to {model_export_path}")